<a href="https://colab.research.google.com/github/Suhail-Ahmed7/flyrank-ml-internship-suhail/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Content Results Curve

The FlyRank research report observed that content health increased during
the early life of a page, peaked in the 61–90 day age group, and then
generally weakened as content became older.

The reported average health score was:

- 37.2 for pages aged 61–90 days
- 31.0 for pages aged 181–270 days
- 29.6 for pages aged 271–365 days

The 365+ day group increased again to 35.5, showing that the relationship
was not a simple continuous decline.

#### Where the measurement comes from

The explanatory variable is the page's content age, divided into age
buckets.

The outcome being compared is FlyRank's internal health score. This is a
composite metric created by FlyRank and is not an official Google ranking
metric.

#### Methodology question

Were pages compared only by age, or were important differences such as
client, content type, initial traffic, refresh history, topic and search
demand also controlled?

A grouped comparison within the same client and content type would help
show whether the age pattern remains after reducing these differences.

#### Constructive interpretation

The reported numbers support an observed association between content age
and health score in this portfolio. They do not prove that simply becoming
older causes a page's performance to decline.

### Finding 2 — The Freshness Multiplier

The FlyRank research report compared page growth and decline across
different freshness windows.

The reported growth-to-decline ratios were:

- 1.0:1 for pages updated 0–30 days ago
- 5.4:1 for pages updated 31–90 days ago
- 2.0:1 for pages updated 91–180 days ago
- 3.0:1 for pages updated 181–360 days ago
- 37.2:1 for pages in the 361+ day group

The 361+ group had only 802 pages and only 21 declining pages, so its
very large ratio should be interpreted carefully.

#### Where the measurement comes from

The explanatory variable is the number of days since a page was last
updated, divided into freshness windows.

The outcome compares the number of pages classified as growing with the
number classified as declining.

The growth and decline labels come from changes in measured page
performance across the report's comparison period.

#### Methodology question

Were pages selected for updates randomly, or were stronger and more
important pages more likely to be refreshed?

If teams deliberately chose promising pages for updates, part of the
observed difference may come from selection bias rather than the update
itself.

A stronger validation design would compare similar refreshed and stale
pages within the same client, content type, age group and previous
traffic range.

#### Constructive interpretation

The report shows an observed association between freshness windows and
page growth in this portfolio.

The result supports using freshness as a decision-support signal for
reviewing pages, but it does not prove that refreshing any page will
automatically cause growth.

In [2]:
%pip -q install duckdb huggingface_hub

import duckdb
import numpy as np
import pandas as pd
import sklearn

from google.colab import userdata

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 150)

# Read the Hugging Face token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

# Connect DuckDB to the FlyRank warehouse.
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB_MAR_TABLE = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    "])"
)

print("Connected successfully to the FlyRank warehouse.")
print("scikit-learn version:", sklearn.__version__)

Connected successfully to the FlyRank warehouse.
scikit-learn version: 1.6.1


In [3]:
# Build one row per webpage.
# February contains the model inputs.
# March is used only to create the later outcome label.

model_frame = con.execute(
    f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            month,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_data_available IS TRUE
                    THEN report_date
                END
            ) AS gsc_observed_days,

            COUNT(
                DISTINCT CASE
                    WHEN ga4_data_available IS TRUE
                    THEN report_date
                END
            ) AS ga4_observed_days,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_impressions,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_clicks,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN gsc_data_available IS TRUE
                        THEN COALESCE(gsc_impressions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS weighted_avg_position,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN ga4_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_engaged_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN ga4_data_available IS TRUE
                        THEN COALESCE(ga4_sessions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS engagement_rate

        FROM {FEB_MAR_TABLE}

        WHERE month IN ('2026-02', '2026-03')

        GROUP BY
            client_hash_id,
            content_hash_id,
            month
    ),

    february AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_observed_days AS feb_gsc_days,
            ga4_observed_days AS feb_ga4_days,
            avg_daily_impressions AS feb_avg_daily_impressions,
            avg_daily_clicks AS feb_avg_daily_clicks,
            weighted_avg_position AS feb_avg_position,
            avg_daily_sessions AS feb_avg_daily_sessions,
            engagement_rate AS feb_engagement_rate
        FROM monthly
        WHERE month = '2026-02'
    ),

    march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_observed_days AS march_gsc_days,
            avg_daily_clicks AS march_avg_daily_clicks
        FROM monthly
        WHERE month = '2026-03'
    )

    SELECT
        feb.client_hash_id,
        feb.content_hash_id,
        '2026-02' AS decision_month,

        feb.feb_avg_daily_impressions,
        feb.feb_avg_daily_clicks,
        feb.feb_avg_position,
        feb.feb_avg_daily_sessions,
        feb.feb_engagement_rate,

        mar.march_avg_daily_clicks,

        CASE
            WHEN mar.march_avg_daily_clicks
                 < feb.feb_avg_daily_clicks
            THEN 1
            ELSE 0
        END AS declined_next_month

    FROM february AS feb

    INNER JOIN march AS mar
        ON feb.client_hash_id = mar.client_hash_id
        AND feb.content_hash_id = mar.content_hash_id

    WHERE feb.feb_gsc_days > 0
      AND feb.feb_ga4_days > 0
      AND mar.march_gsc_days > 0
      AND feb.feb_avg_daily_impressions IS NOT NULL
      AND feb.feb_avg_daily_clicks IS NOT NULL
      AND feb.feb_avg_position IS NOT NULL
      AND feb.feb_avg_daily_sessions IS NOT NULL
      AND feb.feb_engagement_rate IS NOT NULL
      AND mar.march_avg_daily_clicks IS NOT NULL
    """
).df()

print("Modelling rows:", len(model_frame))

print("\nLabel counts:")
print(model_frame["declined_next_month"].value_counts())

print("\nLabel percentages:")
print(
    model_frame["declined_next_month"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(model_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modelling rows: 23742

Label counts:
declined_next_month
0    14391
1     9351
Name: count, dtype: int64

Label percentages:
declined_next_month
0    60.61
1    39.39
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,decision_month,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,march_avg_daily_clicks,declined_next_month
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,2026-02,34.178571,0.000000,13.073145,2.0,0.0,0.068966,0
1,client_e547b89c05043229,content_4c1e972bec56132e,2026-02,102.928571,0.535714,10.403539,1.2,0.0,0.310345,1
2,client_e547b89c05043229,content_2e296120acb03e93,2026-02,23.571429,0.000000,61.680303,1.0,0.5,0.000000,0
3,client_e547b89c05043229,content_38b6c1a9aa29f801,2026-02,37.964286,0.000000,54.554092,1.0,0.0,0.000000,0
4,client_e547b89c05043229,content_f338440914b1ab00,2026-02,45.392857,0.035714,13.079465,1.0,0.0,0.206897,0


In [4]:
# Create the clean modelling table.
model_df = model_frame.copy()

# Keep rows with usable February information.
model_df = model_df[
    (model_df["feb_avg_daily_impressions"] > 0)
    & (model_df["feb_avg_daily_clicks"] >= 0)
    & (model_df["feb_avg_position"] > 0)
].copy()

# Create February click-through rate.
model_df["feb_ctr"] = (
    model_df["feb_avg_daily_clicks"]
    / model_df["feb_avg_daily_impressions"]
)

model_df["feb_ctr"] = model_df["feb_ctr"].replace(
    [np.inf, -np.inf],
    np.nan,
)

# February-only model features.
feature_columns = [
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
    "feb_ctr",
]

target_column = "declined_next_month"
group_column = "client_hash_id"

# Remove rows with missing values.
model_df = model_df.dropna(
    subset=feature_columns + [target_column, group_column]
).copy()

X = model_df[feature_columns].copy()
y = model_df[target_column].astype(int).copy()
groups = model_df[group_column].copy()

print("Usable modelling rows:", len(model_df))
print("Number of features:", len(feature_columns))
print("Unique clients:", groups.nunique())

print("\nFeatures:")
for feature in feature_columns:
    print("-", feature)

print("\nTarget percentages:")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissing values:")
print(X.isna().sum())

Usable modelling rows: 23703
Number of features: 6
Unique clients: 19

Features:
- feb_avg_daily_impressions
- feb_avg_daily_clicks
- feb_avg_position
- feb_avg_daily_sessions
- feb_engagement_rate
- feb_ctr

Target percentages:
declined_next_month
0    60.55
1    39.45
Name: proportion, dtype: float64

Missing values:
feb_avg_daily_impressions    0
feb_avg_daily_clicks         0
feb_avg_position             0
feb_avg_daily_sessions       0
feb_engagement_rate          0
feb_ctr                      0
dtype: int64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation plan

I will evaluate the same Logistic Regression model using two different
split designs.

#### Before — random row split

The first evaluation randomly separates individual webpage rows into
training and testing sets.

This design may be optimistic because pages belonging to the same client
can appear in both sets. The model may benefit from client-specific
patterns that are shared across those pages.

#### After — client-grouped split

The second evaluation separates complete clients using
`client_hash_id`.

All pages belonging to one client remain entirely in either training or
testing. The same client cannot appear in both sets.

This grouped split provides a more honest test of whether the model can
generalize to clients it did not see during training.

#### Fair comparison

Both evaluations will use:

- The same six February-only features
- The same `declined_next_month` target
- The same Logistic Regression pipeline
- The same random seed
- Precision@20
- Precision@50
- Average precision
- The test-set base rate

The difference between the two evaluations will show whether a random
row split gives a more optimistic result than a client-grouped split.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42


def precision_at_k(y_true, probabilities, k):
    top_k_positions = np.argsort(probabilities)[::-1][:k]
    return y_true.iloc[top_k_positions].mean()


# Randomly split individual webpage rows.
random_train_index, random_test_index = train_test_split(
    model_df.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_random = X.loc[random_train_index]
X_test_random = X.loc[random_test_index]

y_train_random = y.loc[random_train_index]
y_test_random = y.loc[random_test_index]

# Create and train Logistic Regression.
random_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_model.fit(X_train_random, y_train_random)

random_probabilities = random_model.predict_proba(
    X_test_random
)[:, 1]

random_base_rate = y_test_random.mean()
random_p20 = precision_at_k(
    y_test_random.reset_index(drop=True),
    random_probabilities,
    20,
)
random_p50 = precision_at_k(
    y_test_random.reset_index(drop=True),
    random_probabilities,
    50,
)
random_ap = average_precision_score(
    y_test_random,
    random_probabilities,
)

# Check whether the same clients appear in both sets.
random_train_clients = set(
    model_df.loc[random_train_index, group_column]
)
random_test_clients = set(
    model_df.loc[random_test_index, group_column]
)

random_client_overlap = (
    random_train_clients & random_test_clients
)

print("Random row split")
print("----------------")
print("Training rows:", len(X_train_random))
print("Testing rows:", len(X_test_random))
print("Clients appearing in both sets:", len(random_client_overlap))

print("\nResults")
print("Base rate:", f"{random_base_rate:.2%}")
print("Precision@20:", f"{random_p20:.2%}")
print("Precision@50:", f"{random_p50:.2%}")
print("Average precision:", f"{random_ap:.4f}")

Random row split
----------------
Training rows: 18962
Testing rows: 4741
Clients appearing in both sets: 16

Results
Base rate: 39.44%
Precision@20: 90.00%
Precision@50: 86.00%
Average precision: 0.6722


In [6]:
from sklearn.model_selection import GroupShuffleSplit

# Split complete clients instead of individual rows.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

group_train_positions, group_test_positions = next(
    group_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train_grouped = X.iloc[group_train_positions]
X_test_grouped = X.iloc[group_test_positions]

y_train_grouped = y.iloc[group_train_positions]
y_test_grouped = y.iloc[group_test_positions]

grouped_train_clients = set(
    groups.iloc[group_train_positions]
)

grouped_test_clients = set(
    groups.iloc[group_test_positions]
)

grouped_client_overlap = (
    grouped_train_clients & grouped_test_clients
)

# Train the same Logistic Regression pipeline.
grouped_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

grouped_model.fit(
    X_train_grouped,
    y_train_grouped,
)

grouped_probabilities = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_base_rate = y_test_grouped.mean()

grouped_p20 = precision_at_k(
    y_test_grouped.reset_index(drop=True),
    grouped_probabilities,
    20,
)

grouped_p50 = precision_at_k(
    y_test_grouped.reset_index(drop=True),
    grouped_probabilities,
    50,
)

grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_probabilities,
)

print("Client-grouped split")
print("--------------------")
print("Training rows:", len(X_train_grouped))
print("Testing rows:", len(X_test_grouped))
print("Training clients:", len(grouped_train_clients))
print("Testing clients:", len(grouped_test_clients))
print("Clients appearing in both sets:", len(grouped_client_overlap))

print("\nResults")
print("Base rate:", f"{grouped_base_rate:.2%}")
print("Precision@20:", f"{grouped_p20:.2%}")
print("Precision@50:", f"{grouped_p50:.2%}")
print("Average precision:", f"{grouped_ap:.4f}")

Client-grouped split
--------------------
Training rows: 22032
Testing rows: 1671
Training clients: 15
Testing clients: 4
Clients appearing in both sets: 0

Results
Base rate: 23.22%
Precision@20: 100.00%
Precision@50: 98.00%
Average precision: 0.8619


In [7]:
comparison_table = pd.DataFrame(
    {
        "Validation design": [
            "Random row split",
            "Client-grouped split",
        ],
        "Test rows": [
            len(X_test_random),
            len(X_test_grouped),
        ],
        "Test clients": [
            len(random_test_clients),
            len(grouped_test_clients),
        ],
        "Client overlap": [
            len(random_client_overlap),
            len(grouped_client_overlap),
        ],
        "Base rate": [
            random_base_rate,
            grouped_base_rate,
        ],
        "Precision@20": [
            random_p20,
            grouped_p20,
        ],
        "Precision@50": [
            random_p50,
            grouped_p50,
        ],
        "Average precision": [
            random_ap,
            grouped_ap,
        ],
    }
)

formatted_comparison = comparison_table.copy()

for column in [
    "Base rate",
    "Precision@20",
    "Precision@50",
]:
    formatted_comparison[column] = (
        formatted_comparison[column]
        .mul(100)
        .round(2)
        .astype(str)
        + "%"
    )

formatted_comparison["Average precision"] = (
    formatted_comparison["Average precision"].round(4)
)

display(formatted_comparison)

,Validation design,Test rows,Test clients,Client overlap,Base rate,Precision@20,Precision@50,Average precision
0,Random row split,4741,16,16,39.44%,90.0%,86.0%,0.6722
1,Client-grouped split,1671,4,0,23.22%,100.0%,98.0%,0.8619


### Before-and-after interpretation

The random row split placed pages from the same clients in both the
training and testing sets. All 16 clients in the random test set also
appeared in the training set.

The client-grouped split had zero client overlap. Its four test clients
were completely unseen during model training.

The random split produced:

- Base rate: 39.44%
- Precision@20: 90%
- Precision@50: 86%
- Average precision: 0.6722

The client-grouped split produced:

- Base rate: 23.22%
- Precision@20: 100%
- Precision@50: 98%
- Average precision: 0.8619

The grouped result was stronger in this particular split. However, this
does not prove that grouped validation always improves model performance.

The two test sets contain different clients, different numbers of rows
and different positive-label base rates. Therefore, the metric difference
may partly reflect the composition of the test clients.

The main validation improvement is not the higher score. The important
improvement is that the grouped split evaluates the model on clients it
did not see during training.

The result provides directional evidence that the February features can
rank some unseen-client pages that later decline. Because only four clients
were included in the grouped test set, the result should not be treated as
a universal performance estimate.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit plan

I will inspect the final feature set for four possible problems:

1. **Future-data leakage**  
   Every model feature must come from February. March data may only be
   used to create the later outcome label.

2. **Target leakage**  
   No feature should directly contain the March click result or the
   `declined_next_month` label.

3. **Identifier leakage**  
   Client and content identifiers must not be included as model features.

4. **Suspiciously strong features**  
   I will inspect feature coefficients and compare model performance
   after removing the strongest feature.

My final model features are:

- February average daily impressions
- February average daily clicks
- February average position
- February average daily sessions
- February engagement rate
- February click-through rate

The March average daily clicks column is retained only for label creation
and is not included in the model input.

In [8]:
# Check the final model features for obvious leakage.

future_features = [
    feature for feature in feature_columns
    if "march" in feature.lower()
]

target_features = [
    feature for feature in feature_columns
    if feature == target_column
    or "declined_next_month" in feature.lower()
]

identifier_features = [
    feature for feature in feature_columns
    if feature in [
        "client_hash_id",
        "content_hash_id",
        "decision_month",
    ]
    or "hash_id" in feature.lower()
]

non_february_features = [
    feature for feature in feature_columns
    if not feature.startswith("feb_")
]

print("Leakage audit")
print("-------------")
print("Future/March features:", future_features)
print("Target-derived features:", target_features)
print("Identifier features:", identifier_features)
print("Features not clearly from February:", non_february_features)

print("\nAudit checks")
print("No future-data leakage:", len(future_features) == 0)
print("No target leakage:", len(target_features) == 0)
print("No identifier leakage:", len(identifier_features) == 0)
print("All features are February-only:", len(non_february_features) == 0)

assert len(future_features) == 0
assert len(target_features) == 0
assert len(identifier_features) == 0
assert len(non_february_features) == 0

print("\nAll basic leakage checks passed.")

Leakage audit
-------------
Future/March features: []
Target-derived features: []
Identifier features: []
Features not clearly from February: []

Audit checks
No future-data leakage: True
No target leakage: True
No identifier leakage: True
All features are February-only: True

All basic leakage checks passed.


In [9]:
# Inspect Logistic Regression feature strength.
grouped_coefficients = (
    grouped_model
    .named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Coefficient": grouped_coefficients,
        "Absolute coefficient": np.abs(grouped_coefficients),
    }
).sort_values(
    "Absolute coefficient",
    ascending=False,
)

display(
    coefficient_table.reset_index(drop=True)
)

strongest_feature = coefficient_table.iloc[0]["Feature"]

print(
    "Strongest feature:",
    strongest_feature,
)

,Feature,Coefficient,Absolute coefficient
0,feb_ctr,3.875175,3.875175
1,feb_avg_daily_impressions,0.373530,0.373530
2,feb_avg_daily_clicks,-0.156003,0.156003
3,feb_avg_position,-0.102046,0.102046
4,feb_engagement_rate,0.013873,0.013873
5,feb_avg_daily_sessions,0.011596,0.011596


Strongest feature: feb_ctr


In [10]:
# Re-train the grouped model without the strongest feature.

reduced_features = [
    feature
    for feature in feature_columns
    if feature != strongest_feature
]

X_reduced = model_df[reduced_features].copy()

X_train_reduced = X_reduced.iloc[group_train_positions]
X_test_reduced = X_reduced.iloc[group_test_positions]

reduced_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

reduced_model.fit(
    X_train_reduced,
    y_train_grouped,
)

reduced_probabilities = reduced_model.predict_proba(
    X_test_reduced
)[:, 1]

reduced_p20 = precision_at_k(
    y_test_grouped.reset_index(drop=True),
    reduced_probabilities,
    20,
)

reduced_p50 = precision_at_k(
    y_test_grouped.reset_index(drop=True),
    reduced_probabilities,
    50,
)

reduced_ap = average_precision_score(
    y_test_grouped,
    reduced_probabilities,
)

feature_test_table = pd.DataFrame(
    {
        "Model version": [
            "All six features",
            "Without feb_ctr",
        ],
        "Precision@20": [
            grouped_p20,
            reduced_p20,
        ],
        "Precision@50": [
            grouped_p50,
            reduced_p50,
        ],
        "Average precision": [
            grouped_ap,
            reduced_ap,
        ],
    }
)

for column in ["Precision@20", "Precision@50"]:
    feature_test_table[column] = (
        feature_test_table[column]
        .mul(100)
        .round(2)
        .astype(str)
        + "%"
    )

feature_test_table["Average precision"] = (
    feature_test_table["Average precision"].round(4)
)

display(feature_test_table)

print("Removed feature:", strongest_feature)

,Model version,Precision@20,Precision@50,Average precision
0,All six features,100.0%,98.0%,0.8619
1,Without feb_ctr,40.0%,36.0%,0.4231


Removed feature: feb_ctr


### Strongest-feature audit

`feb_ctr` was the strongest model feature. Its standardized Logistic
Regression coefficient was 3.8752, much larger than the coefficients of
the other five features.

When `feb_ctr` was removed, performance changed from:

- Precision@20: 100% to 40%
- Precision@50: 98% to 36%
- Average precision: 0.8619 to 0.4231

This large decrease shows that the model depends heavily on February CTR.

The result is not, by itself, evidence of future-data leakage because
`feb_ctr` is calculated only from February clicks and February
impressions, while the outcome is measured using March clicks.

However, the dependence is important. The strong grouped result should
not be described as broad model intelligence because most of the ranking
signal comes from one derived feature.

A stronger future audit would repeat the grouped evaluation across
several client folds and check whether the importance of February CTR
remains stable across different unseen clients.

### Leakage-audit conclusion

The final model contains:

- No March or future-period features
- No target or target-derived feature
- No client or content identifiers
- No existing decision flag
- Six features available at the February decision point

The basic leakage checks passed. The main limitation is feature
concentration rather than confirmed leakage: the model relies heavily on
February CTR.

In [14]:
from sklearn.metrics import confusion_matrix

# Build a review table for the honest client-grouped test set.
# Reset the index so every array aligns by row position.
error_table = (
    model_df.iloc[group_test_positions][
        [
            "content_hash_id",
            "feb_avg_daily_impressions",
            "feb_avg_daily_clicks",
            "feb_avg_position",
            "feb_avg_daily_sessions",
            "feb_engagement_rate",
            "feb_ctr",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

error_table["actual_label"] = (
    y_test_grouped
    .reset_index(drop=True)
    .astype(int)
)

error_table["predicted_probability"] = grouped_probabilities

error_table["predicted_label"] = (
    error_table["predicted_probability"] >= 0.50
).astype(int)

conditions = [
    (
        (error_table["actual_label"] == 1)
        & (error_table["predicted_label"] == 1)
    ),
    (
        (error_table["actual_label"] == 0)
        & (error_table["predicted_label"] == 0)
    ),
    (
        (error_table["actual_label"] == 0)
        & (error_table["predicted_label"] == 1)
    ),
    (
        (error_table["actual_label"] == 1)
        & (error_table["predicted_label"] == 0)
    ),
]

outcomes = [
    "True positive",
    "True negative",
    "False positive",
    "False negative",
]

error_table["outcome"] = np.select(
    conditions,
    outcomes,
    default="Unknown",
)

tn, fp, fn, tp = confusion_matrix(
    error_table["actual_label"],
    error_table["predicted_label"],
    labels=[0, 1],
).ravel()

print("Grouped test-set error summary")
print("------------------------------")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print("\nOutcome counts:")
print(error_table["outcome"].value_counts())

print("\nMissing actual labels:", error_table["actual_label"].isna().sum())

Grouped test-set error summary
------------------------------
True negatives: 1252
False positives: 31
False negatives: 216
True positives: 172

Outcome counts:
outcome
True negative     1252
False negative     216
True positive      172
False positive      31
Name: count, dtype: int64

Missing actual labels: 0


In [15]:
# Select three different model failure examples.

false_positives = error_table[
    error_table["outcome"] == "False positive"
].copy()

false_negatives = error_table[
    error_table["outcome"] == "False negative"
].copy()

# Case 1: The false positive with the highest predicted probability.
high_confidence_fp = (
    false_positives
    .sort_values(
        "predicted_probability",
        ascending=False,
    )
    .head(1)
    .copy()
)

high_confidence_fp["case_type"] = (
    "High-confidence false positive"
)

# Case 2: The false negative closest to the 0.50 threshold.
borderline_fn = (
    false_negatives
    .assign(
        distance_from_threshold=lambda frame:
        abs(frame["predicted_probability"] - 0.50)
    )
    .sort_values("distance_from_threshold")
    .head(1)
    .copy()
)

borderline_fn["case_type"] = (
    "Borderline false negative"
)

# Case 3: A false negative with high February visibility.
remaining_false_negatives = false_negatives.drop(
    index=borderline_fn.index,
    errors="ignore",
)

high_visibility_fn = (
    remaining_false_negatives
    .sort_values(
        "feb_avg_daily_impressions",
        ascending=False,
    )
    .head(1)
    .copy()
)

high_visibility_fn["case_type"] = (
    "High-visibility false negative"
)

failure_examples = pd.concat(
    [
        high_confidence_fp,
        borderline_fn,
        high_visibility_fn,
    ],
    ignore_index=True,
)

failure_examples = failure_examples[
    [
        "case_type",
        "content_hash_id",
        "actual_label",
        "predicted_label",
        "predicted_probability",
        "feb_avg_daily_impressions",
        "feb_avg_daily_clicks",
        "feb_ctr",
        "feb_avg_position",
        "feb_avg_daily_sessions",
        "feb_engagement_rate",
    ]
]

display(failure_examples.round(6))

,case_type,content_hash_id,actual_label,predicted_label,predicted_probability,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_ctr,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate
0,High-confidence false positive,content_d95b42029414d8f7,0,1,0.999086,5.500000,0.321429,0.058442,6.000000,1.142857,0.0
1,Borderline false negative,content_c697dd224ba7d717,1,0,0.498504,4.615385,0.038462,0.008333,29.858333,1.000000,0.0
2,High-visibility false negative,content_c04060d322f34d83,1,0,0.419409,232.178571,0.214286,0.000923,7.162590,1.666667,0.2


### Real failure examples

I reviewed three incorrect predictions from the honest client-grouped
test set.

#### Case 1 — High-confidence false positive

The model assigned a decline probability of 0.9991, but the page did not
decline.

This page had:

- 5.50 average daily impressions
- 0.32 average daily clicks
- February CTR of 5.84%
- Average position of 6.00

The model appears to have treated its relatively high February CTR as a
strong decline signal. However, the page had very low traffic volume.
A small change in clicks can create a large CTR change for a low-volume
page, so the prediction may have been unstable.

#### Case 2 — Borderline false negative

The model assigned a decline probability of 0.4985, just below the 0.50
classification threshold, but the page later declined.

This was a borderline decision rather than a confident mistake. A small
change in the threshold would have changed the predicted class.

This example shows that binary labels hide uncertainty around the decision
boundary. The probability ranking may be more useful than treating every
0.50 threshold decision as certain.

#### Case 3 — High-visibility false negative

The model assigned a decline probability of 0.4194, but the page later
declined.

This page had:

- 232.18 average daily impressions
- 0.21 average daily clicks
- February CTR of 0.09%
- Average position of 7.16

Despite its high visibility and low CTR, the model did not assign a
probability above 0.50. This suggests that February CTR alone does not
capture every way that a page can decline.

Other information such as search-demand changes, content updates,
seasonality, page type and client context may be needed.

### Error-analysis conclusion

At the 0.50 threshold, the grouped test set contained:

- 31 false positives
- 216 false negatives
- 172 true positives
- 1,252 true negatives

The model missed more declining pages than it incorrectly flagged.

Therefore, this model is more suitable as a ranked decision-support queue
than as an automatic system that makes final decline decisions.

## 4. Claim rewrite

### Original claim

The Logistic Regression model accurately predicts which webpages will
decline next month and can be used to identify declining content.

### Why this claim is too strong

This sentence is stronger than the evidence because:

- The model was tested on only four unseen clients.
- Most of its ranking performance depended on February CTR.
- The result comes from one grouped train/test split.
- The model produced 31 false positives and 216 false negatives.
- The analysis shows association and predictive ranking, not causation.
- The 0.50 classification threshold missed many declining pages.

The model does not prove that its features cause future decline, and the
measured result should not be treated as universal performance.

### Safer rewritten claim

In this dataset and client-grouped holdout, the Logistic Regression model
used February page-performance features to rank pages by their measured
risk of lower average daily clicks in March.

On four unseen test clients, the model achieved:

- Precision@20 of 100%
- Precision@50 of 98%
- Average precision of 0.8619
- Test-set base rate of 23.22%

The result provides directional evidence that these features, especially
February CTR, may support a ranked review queue for identifying pages that
deserve attention.

The model should be used as decision support rather than as an automatic
final decision system. Its performance should be validated across more
client-grouped folds and future time periods before broader use.

### Final evidence statement

I observed strong ranking performance on this specific grouped holdout.
I did not prove that CTR causes future decline or that the same performance
will occur for every client or time period.

In [16]:
# Final notebook audit.

audit_results = {
    "Dataset contains rows": len(model_df) > 0,
    "Six model features used": len(feature_columns) == 6,
    "No future features": len(future_features) == 0,
    "No target leakage": len(target_features) == 0,
    "No identifier features": len(identifier_features) == 0,
    "Random split contains client overlap": len(random_client_overlap) > 0,
    "Grouped split contains zero overlap": len(grouped_client_overlap) == 0,
    "Base rate reported": grouped_base_rate >= 0,
    "Three failure cases reviewed": len(failure_examples) == 3,
    "Fixed random seed used": RANDOM_STATE == 42,
}

print("Final validation audit")
print("----------------------")

for check, passed in audit_results.items():
    print(f"{check}: {passed}")

assert all(audit_results.values())

print("\nAll final audit checks passed.")

Final validation audit
----------------------
Dataset contains rows: True
Six model features used: True
No future features: True
No target leakage: True
No identifier features: True
Random split contains client overlap: True
Grouped split contains zero overlap: True
Base rate reported: True
Three failure cases reviewed: True
Fixed random seed used: True

All final audit checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.